GPT4o Results

In [ ]:
import json
import csv
import re
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────

# JSON file with LLM test execution results on breaking version (v2)
LLM_RESULTS_JSON = "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"

# CSV file with BUMP ground-truth breaking change metadata
BUMP_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/BUMPErrorLogs/RQ4_resultsBUMP.csv"

# Directory containing test execution logs for breaking version
LLM_LOGS_DIR = "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/bre/logs"

# Output CSV — one row per instance where LLM detected a BC
OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv"

# Hardcoded experiment metadata — update these before each run
MODEL_NAME      = "GPT-4o"    # e.g. "GPT-4o", "Qwen-480B", "GPT-OSS-120b"
CONTEXT_VARIANT = "Minimal"   # e.g. "Minimal", "Method", "Class"

# ──────────────────────────────────────────────────────────────────────────────


EXCEPTION_PATTERNS = [
    r'(?:^|\s)([\w\.]+(?:Exception|Error))(?:\s|:|\(|$)',
    r'Caused by:\s+([\w\.]+(?:Exception|Error))',
    r'(?:throws|threw|raised)\s+([\w\.]+(?:Exception|Error))',
]

EXCLUDED = {'Error', 'Exception', 'MojoFailureException', 'MojoExecutionException'}


def parse_bump_errors(raw: str) -> set:
    """
    Parse the exception_types column from BUMP CSV into a set of
    short exception class names (e.g. 'NoClassDefFoundError').
    Pipe-separated values are split and Maven-specific exceptions are excluded.
    """
    if not raw or str(raw).strip() in ('', 'nan'):
        return set()
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        # Extract short class name: java.lang.NoClassDefFoundError → NoClassDefFoundError
        result.add(e.split('.')[-1])
    return result


def parse_log_errors(log_path: Path) -> set:
    """
    Extract all exception/error class names from a test execution log file.
    Returns a set of short class names (e.g. 'NoClassDefFoundError').
    """
    if not log_path.exists():
        return set()
    try:
        content = log_path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return set()
    found = set()
    for pattern in EXCEPTION_PATTERNS:
        for match in re.findall(pattern, content, re.MULTILINE):
            name = match.strip().split('.')[-1]
            if name and name not in EXCLUDED and ('Exception' in name or 'Error' in name):
                if not name.startswith('test'):
                    found.add(name)
    return found


def load_bump(bump_csv: str) -> dict:
    """
    Load BUMP CSV into a dict keyed by instance ID (custom_id).
    Returns per-instance: bump_errors (set), bump_errors_raw (str), failure_category (str).
    """
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            data[row['custom_id']] = {
                'bump_errors':      parse_bump_errors(row.get('exception_types', '')),
                'bump_errors_raw':  row.get('exception_types', ''),
                'failure_category': row.get('failureCategory', ''),
            }
    return data


def load_llm(llm_json: str) -> dict:
    """
    Load LLM test execution results JSON.
    Returns the 'results' dict keyed by instance ID.
    """
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    bump = load_bump(BUMP_CSV)
    llm  = load_llm(LLM_RESULTS_JSON)
    logs = Path(LLM_LOGS_DIR)

    rows = []

    for instance_id, instance_data in llm.items():
        failed_tests = instance_data.get('tests', {}).get('failed', [])

        # Only process instances where LLM detected a BC (has failed tests on v2)
        if not failed_tests:
            continue
        if instance_id not in bump:
            print(f"Warning: {instance_id} not in BUMP CSV — skipping")
            continue

        bump_errors = bump[instance_id]['bump_errors']

        # Collect all exception types from failed test logs
        llm_errors = set()
        for test in failed_tests:
            if test.get('result_type') == 'transplant_issue':
                continue
            test_file = test.get('file', '')
            if not test_file:
                continue
            log_path = logs / f"{instance_id}_{test_file}_breaking_single.log"
            llm_errors.update(parse_log_errors(log_path))

        matched = bump_errors & llm_errors
        missed  = bump_errors - llm_errors
        new     = llm_errors  - bump_errors

        match_rate = round(len(matched) / len(bump_errors) * 100, 1) if bump_errors else 0.0

        rows.append({
            'model':               MODEL_NAME,
            'context_variant':     CONTEXT_VARIANT,
            'instance':            instance_id,
            'bump_bc_failure_category': bump[instance_id]['failure_category'],
            'bump_bc_errors_raw':  bump[instance_id]['bump_errors_raw'],
            'bump_bc_errors':      '|'.join(sorted(bump_errors)),
            'llm_detected_errors': '|'.join(sorted(llm_errors)),
            'matched_errors':      '|'.join(sorted(matched)),
            'missed_errors':       '|'.join(sorted(missed)),
            'new_errors':          '|'.join(sorted(new)),
            'bump_total':          len(bump_errors),
            'llm_total':           len(llm_errors),
            'matched_count':       len(matched),
            'missed_count':        len(missed),
            'new_count':           len(new),
            'match_rate_%':        match_rate,
            'num_failed_tests':    len(failed_tests),
        })

     # Write CSV — append if file exists, write with header if new
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'model', 'context_variant',
        'instance', 'bump_bc_failure_category', 'bump_bc_errors_raw',
        'bump_bc_errors', 'llm_detected_errors', 'matched_errors', 'missed_errors', 'new_errors',
        'bump_total', 'llm_total', 'matched_count', 'missed_count', 'new_count',
        'match_rate_%', 'num_failed_tests',
    ]
    file_exists = Path(OUTPUT_CSV).exists()
    with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()  # only write header if file is new
        writer.writerows(rows)

    print(f"Done. {len(rows)} detected instances written to {OUTPUT_CSV}")
    print(f"Model: {MODEL_NAME} | Context Variant: {CONTEXT_VARIANT}")


if __name__ == "__main__":
    main()

Done. 17 detected instances written to /Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv
Model: GPT-4o | Context Variant: Minimal


The count is not matching for method in gpt4o

In [ ]:
import json
import csv
import re
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────

# JSON file with LLM test execution results on breaking version (v2)
LLM_RESULTS_JSON = "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"

# CSV file with BUMP ground-truth breaking change metadata
BUMP_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/BUMPErrorLogs/RQ4_resultsBUMP.csv"

# Directory containing test execution logs for breaking version
LLM_LOGS_DIR = "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/bre/logs"

# Output CSV — one row per instance where LLM detected a BC
OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv"

# Hardcoded experiment metadata — update these before each run
MODEL_NAME      = "GPT-4o"    # e.g. "GPT-4o", "Qwen-480B", "GPT-OSS-120b"
CONTEXT_VARIANT = "Method"   # e.g. "Minimal", "Method", "Class"

# ──────────────────────────────────────────────────────────────────────────────


EXCEPTION_PATTERNS = [
    r'(?:^|\s)([\w\.]+(?:Exception|Error))(?:\s|:|\(|$)',
    r'Caused by:\s+([\w\.]+(?:Exception|Error))',
    r'(?:throws|threw|raised)\s+([\w\.]+(?:Exception|Error))',
]

EXCLUDED = {'Error', 'Exception', 'MojoFailureException', 'MojoExecutionException'}


def parse_bump_errors(raw: str) -> set:
    """
    Parse the exception_types column from BUMP CSV into a set of
    short exception class names (e.g. 'NoClassDefFoundError').
    Pipe-separated values are split and Maven-specific exceptions are excluded.
    """
    if not raw or str(raw).strip() in ('', 'nan'):
        return set()
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        # Extract short class name: java.lang.NoClassDefFoundError → NoClassDefFoundError
        result.add(e.split('.')[-1])
    return result


def parse_log_errors(log_path: Path) -> set:
    """
    Extract all exception/error class names from a test execution log file.
    Returns a set of short class names (e.g. 'NoClassDefFoundError').
    """
    if not log_path.exists():
        return set()
    try:
        content = log_path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return set()
    found = set()
    for pattern in EXCEPTION_PATTERNS:
        for match in re.findall(pattern, content, re.MULTILINE):
            name = match.strip().split('.')[-1]
            if name and name not in EXCLUDED and ('Exception' in name or 'Error' in name):
                if not name.startswith('test'):
                    found.add(name)
    return found


def load_bump(bump_csv: str) -> dict:
    """
    Load BUMP CSV into a dict keyed by instance ID (custom_id).
    Returns per-instance: bump_errors (set), bump_errors_raw (str), failure_category (str).
    """
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            data[row['custom_id']] = {
                'bump_errors':      parse_bump_errors(row.get('exception_types', '')),
                'bump_errors_raw':  row.get('exception_types', ''),
                'failure_category': row.get('failureCategory', ''),
            }
    return data


def load_llm(llm_json: str) -> dict:
    """
    Load LLM test execution results JSON.
    Returns the 'results' dict keyed by instance ID.
    """
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    bump = load_bump(BUMP_CSV)
    llm  = load_llm(LLM_RESULTS_JSON)
    logs = Path(LLM_LOGS_DIR)

    rows = []

    for instance_id, instance_data in llm.items():
        failed_tests = instance_data.get('tests', {}).get('failed', [])

        # Only process instances where LLM detected a BC (has failed tests on v2)
        if not failed_tests:
            continue
        if instance_id not in bump:
            print(f"Warning: {instance_id} not in BUMP CSV — skipping")
            continue

        bump_errors = bump[instance_id]['bump_errors']

        # Collect all exception types from failed test logs
        llm_errors = set()
        for test in failed_tests:
            if test.get('result_type') == 'transplant_issue':
                continue
            test_file = test.get('file', '')
            if not test_file:
                continue
            log_path = logs / f"{instance_id}_{test_file}_breaking_single.log"
            llm_errors.update(parse_log_errors(log_path))

        matched = bump_errors & llm_errors
        missed  = bump_errors - llm_errors
        new     = llm_errors  - bump_errors

        match_rate = round(len(matched) / len(bump_errors) * 100, 1) if bump_errors else 0.0

        rows.append({
            'model':               MODEL_NAME,
            'context_variant':     CONTEXT_VARIANT,
            'instance':            instance_id,
            'bump_bc_failure_category': bump[instance_id]['failure_category'],
            'bump_bc_errors_raw':  bump[instance_id]['bump_errors_raw'],
            'bump_bc_errors':      '|'.join(sorted(bump_errors)),
            'llm_detected_errors': '|'.join(sorted(llm_errors)),
            'matched_errors':      '|'.join(sorted(matched)),
            'missed_errors':       '|'.join(sorted(missed)),
            'new_errors':          '|'.join(sorted(new)),
            'bump_total':          len(bump_errors),
            'llm_total':           len(llm_errors),
            'matched_count':       len(matched),
            'missed_count':        len(missed),
            'new_count':           len(new),
            'match_rate_%':        match_rate,
            'num_failed_tests':    len(failed_tests),
        })

     # Write CSV — append if file exists, write with header if new
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'model', 'context_variant',
        'instance', 'bump_bc_failure_category', 'bump_bc_errors_raw',
        'bump_bc_errors', 'llm_detected_errors', 'matched_errors', 'missed_errors', 'new_errors',
        'bump_total', 'llm_total', 'matched_count', 'missed_count', 'new_count',
        'match_rate_%', 'num_failed_tests',
    ]
    file_exists = Path(OUTPUT_CSV).exists()
    with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()  # only write header if file is new
        writer.writerows(rows)

    print(f"Done. {len(rows)} detected instances written to {OUTPUT_CSV}")
    print(f"Model: {MODEL_NAME} | Context Variant: {CONTEXT_VARIANT}")


if __name__ == "__main__":
    main()

Done. 13 detected instances written to /Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv
Model: GPT-4o | Context Variant: Method


In [ ]:
import json
import csv
import re
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────

# JSON file with LLM test execution results on breaking version (v2)
LLM_RESULTS_JSON = "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/transplant_results_breaking_single_module.json"

# CSV file with BUMP ground-truth breaking change metadata
BUMP_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/BUMPErrorLogs/RQ4_resultsBUMP.csv"

# Directory containing test execution logs for breaking version
LLM_LOGS_DIR = "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/bre/logs"

# Output CSV — one row per instance where LLM detected a BC
OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv"

# Hardcoded experiment metadata — update these before each run
MODEL_NAME      = "GPT-4o"    # e.g. "GPT-4o", "Qwen-480B", "GPT-OSS-120b"
CONTEXT_VARIANT = "Class"   # e.g. "Minimal", "Method", "Class"

# ──────────────────────────────────────────────────────────────────────────────


EXCEPTION_PATTERNS = [
    r'(?:^|\s)([\w\.]+(?:Exception|Error))(?:\s|:|\(|$)',
    r'Caused by:\s+([\w\.]+(?:Exception|Error))',
    r'(?:throws|threw|raised)\s+([\w\.]+(?:Exception|Error))',
]

EXCLUDED = {'Error', 'Exception', 'MojoFailureException', 'MojoExecutionException'}


def parse_bump_errors(raw: str) -> set:
    """
    Parse the exception_types column from BUMP CSV into a set of
    short exception class names (e.g. 'NoClassDefFoundError').
    Pipe-separated values are split and Maven-specific exceptions are excluded.
    """
    if not raw or str(raw).strip() in ('', 'nan'):
        return set()
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        # Extract short class name: java.lang.NoClassDefFoundError → NoClassDefFoundError
        result.add(e.split('.')[-1])
    return result


def parse_log_errors(log_path: Path) -> set:
    """
    Extract all exception/error class names from a test execution log file.
    Returns a set of short class names (e.g. 'NoClassDefFoundError').
    """
    if not log_path.exists():
        return set()
    try:
        content = log_path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return set()
    found = set()
    for pattern in EXCEPTION_PATTERNS:
        for match in re.findall(pattern, content, re.MULTILINE):
            name = match.strip().split('.')[-1]
            if name and name not in EXCLUDED and ('Exception' in name or 'Error' in name):
                if not name.startswith('test'):
                    found.add(name)
    return found


def load_bump(bump_csv: str) -> dict:
    """
    Load BUMP CSV into a dict keyed by instance ID (custom_id).
    Returns per-instance: bump_errors (set), bump_errors_raw (str), failure_category (str).
    """
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            data[row['custom_id']] = {
                'bump_errors':      parse_bump_errors(row.get('exception_types', '')),
                'bump_errors_raw':  row.get('exception_types', ''),
                'failure_category': row.get('failureCategory', ''),
            }
    return data


def load_llm(llm_json: str) -> dict:
    """
    Load LLM test execution results JSON.
    Returns the 'results' dict keyed by instance ID.
    """
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    bump = load_bump(BUMP_CSV)
    llm  = load_llm(LLM_RESULTS_JSON)
    logs = Path(LLM_LOGS_DIR)

    rows = []

    for instance_id, instance_data in llm.items():
        failed_tests = instance_data.get('tests', {}).get('failed', [])

        # Only process instances where LLM detected a BC (has failed tests on v2)
        if not failed_tests:
            continue
        if instance_id not in bump:
            print(f"Warning: {instance_id} not in BUMP CSV — skipping")
            continue

        bump_errors = bump[instance_id]['bump_errors']

        # Collect all exception types from failed test logs
        llm_errors = set()
        for test in failed_tests:
            if test.get('result_type') == 'transplant_issue':
                continue
            test_file = test.get('file', '')
            if not test_file:
                continue
            log_path = logs / f"{instance_id}_{test_file}_breaking_single.log"
            llm_errors.update(parse_log_errors(log_path))

        matched = bump_errors & llm_errors
        missed  = bump_errors - llm_errors
        new     = llm_errors  - bump_errors

        match_rate = round(len(matched) / len(bump_errors) * 100, 1) if bump_errors else 0.0

        rows.append({
            'model':               MODEL_NAME,
            'context_variant':     CONTEXT_VARIANT,
            'instance':            instance_id,
            'bump_bc_failure_category': bump[instance_id]['failure_category'],
            'bump_bc_errors_raw':  bump[instance_id]['bump_errors_raw'],
            'bump_bc_errors':      '|'.join(sorted(bump_errors)),
            'llm_detected_errors': '|'.join(sorted(llm_errors)),
            'matched_errors':      '|'.join(sorted(matched)),
            'missed_errors':       '|'.join(sorted(missed)),
            'new_errors':          '|'.join(sorted(new)),
            'bump_total':          len(bump_errors),
            'llm_total':           len(llm_errors),
            'matched_count':       len(matched),
            'missed_count':        len(missed),
            'new_count':           len(new),
            'match_rate_%':        match_rate,
            'num_failed_tests':    len(failed_tests),
        })

     # Write CSV — append if file exists, write with header if new
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'model', 'context_variant',
        'instance', 'bump_bc_failure_category', 'bump_bc_errors_raw',
        'bump_bc_errors', 'llm_detected_errors', 'matched_errors', 'missed_errors', 'new_errors',
        'bump_total', 'llm_total', 'matched_count', 'missed_count', 'new_count',
        'match_rate_%', 'num_failed_tests',
    ]
    file_exists = Path(OUTPUT_CSV).exists()
    with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()  # only write header if file is new
        writer.writerows(rows)

    print(f"Done. {len(rows)} detected instances written to {OUTPUT_CSV}")
    print(f"Model: {MODEL_NAME} | Context Variant: {CONTEXT_VARIANT}")


if __name__ == "__main__":
    main()

Done. 27 detected instances written to /Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv
Model: GPT-4o | Context Variant: Class


QWen model results

In [ ]:
import json
import csv
import re
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────

# JSON file with LLM test execution results on breaking version (v2)
LLM_RESULTS_JSON = "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"

# CSV file with BUMP ground-truth breaking change metadata
BUMP_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/BUMPErrorLogs/RQ4_resultsBUMP.csv"

# Directory containing test execution logs for breaking version
LLM_LOGS_DIR = "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/bre/logs"

# Output CSV — one row per instance where LLM detected a BC
OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv"

# Hardcoded experiment metadata — update these before each run
MODEL_NAME      = "Qwen-480B"    # e.g. "GPT-4o", "Qwen-480B", "GPT-OSS-120b"
CONTEXT_VARIANT = "Minimal"   # e.g. "Minimal", "Method", "Class"

# ──────────────────────────────────────────────────────────────────────────────


EXCEPTION_PATTERNS = [
    r'(?:^|\s)([\w\.]+(?:Exception|Error))(?:\s|:|\(|$)',
    r'Caused by:\s+([\w\.]+(?:Exception|Error))',
    r'(?:throws|threw|raised)\s+([\w\.]+(?:Exception|Error))',
]

EXCLUDED = {'Error', 'Exception', 'MojoFailureException', 'MojoExecutionException'}


def parse_bump_errors(raw: str) -> set:
    """
    Parse the exception_types column from BUMP CSV into a set of
    short exception class names (e.g. 'NoClassDefFoundError').
    Pipe-separated values are split and Maven-specific exceptions are excluded.
    """
    if not raw or str(raw).strip() in ('', 'nan'):
        return set()
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        # Extract short class name: java.lang.NoClassDefFoundError → NoClassDefFoundError
        result.add(e.split('.')[-1])
    return result


def parse_log_errors(log_path: Path) -> set:
    """
    Extract all exception/error class names from a test execution log file.
    Returns a set of short class names (e.g. 'NoClassDefFoundError').
    """
    if not log_path.exists():
        return set()
    try:
        content = log_path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return set()
    found = set()
    for pattern in EXCEPTION_PATTERNS:
        for match in re.findall(pattern, content, re.MULTILINE):
            name = match.strip().split('.')[-1]
            if name and name not in EXCLUDED and ('Exception' in name or 'Error' in name):
                if not name.startswith('test'):
                    found.add(name)
    return found


def load_bump(bump_csv: str) -> dict:
    """
    Load BUMP CSV into a dict keyed by instance ID (custom_id).
    Returns per-instance: bump_errors (set), bump_errors_raw (str), failure_category (str).
    """
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            data[row['custom_id']] = {
                'bump_errors':      parse_bump_errors(row.get('exception_types', '')),
                'bump_errors_raw':  row.get('exception_types', ''),
                'failure_category': row.get('failureCategory', ''),
            }
    return data


def load_llm(llm_json: str) -> dict:
    """
    Load LLM test execution results JSON.
    Returns the 'results' dict keyed by instance ID.
    """
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    bump = load_bump(BUMP_CSV)
    llm  = load_llm(LLM_RESULTS_JSON)
    logs = Path(LLM_LOGS_DIR)

    rows = []

    for instance_id, instance_data in llm.items():
        failed_tests = instance_data.get('tests', {}).get('failed', [])

        # Only process instances where LLM detected a BC (has failed tests on v2)
        if not failed_tests:
            continue
        if instance_id not in bump:
            print(f"Warning: {instance_id} not in BUMP CSV — skipping")
            continue

        bump_errors = bump[instance_id]['bump_errors']

        # Collect all exception types from failed test logs
        llm_errors = set()
        for test in failed_tests:
            if test.get('result_type') == 'transplant_issue':
                continue
            test_file = test.get('file', '')
            if not test_file:
                continue
            log_path = logs / f"{instance_id}_{test_file}_breaking_single.log"
            llm_errors.update(parse_log_errors(log_path))

        matched = bump_errors & llm_errors
        missed  = bump_errors - llm_errors
        new     = llm_errors  - bump_errors

        match_rate = round(len(matched) / len(bump_errors) * 100, 1) if bump_errors else 0.0

        rows.append({
            'model':               MODEL_NAME,
            'context_variant':     CONTEXT_VARIANT,
            'instance':            instance_id,
            'bump_bc_failure_category': bump[instance_id]['failure_category'],
            'bump_bc_errors_raw':  bump[instance_id]['bump_errors_raw'],
            'bump_bc_errors':      '|'.join(sorted(bump_errors)),
            'llm_detected_errors': '|'.join(sorted(llm_errors)),
            'matched_errors':      '|'.join(sorted(matched)),
            'missed_errors':       '|'.join(sorted(missed)),
            'new_errors':          '|'.join(sorted(new)),
            'bump_total':          len(bump_errors),
            'llm_total':           len(llm_errors),
            'matched_count':       len(matched),
            'missed_count':        len(missed),
            'new_count':           len(new),
            'match_rate_%':        match_rate,
            'num_failed_tests':    len(failed_tests),
        })

     # Write CSV — append if file exists, write with header if new
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'model', 'context_variant',
        'instance', 'bump_bc_failure_category', 'bump_bc_errors_raw',
        'bump_bc_errors', 'llm_detected_errors', 'matched_errors', 'missed_errors', 'new_errors',
        'bump_total', 'llm_total', 'matched_count', 'missed_count', 'new_count',
        'match_rate_%', 'num_failed_tests',
    ]
    file_exists = Path(OUTPUT_CSV).exists()
    with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()  # only write header if file is new
        writer.writerows(rows)

    print(f"Done. {len(rows)} detected instances written to {OUTPUT_CSV}")
    print(f"Model: {MODEL_NAME} | Context Variant: {CONTEXT_VARIANT}")


if __name__ == "__main__":
    main()

Done. 18 detected instances written to /Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv
Model: Qwen-480B | Context Variant: Minimal


In [ ]:
import json
import csv
import re
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────

# JSON file with LLM test execution results on breaking version (v2)
LLM_RESULTS_JSON = "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"

# CSV file with BUMP ground-truth breaking change metadata
BUMP_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/BUMPErrorLogs/RQ4_resultsBUMP.csv"

# Directory containing test execution logs for breaking version
LLM_LOGS_DIR = "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/bre/logs"

# Output CSV — one row per instance where LLM detected a BC
OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv"

# Hardcoded experiment metadata — update these before each run
MODEL_NAME      = "Qwen-480B"    # e.g. "GPT-4o", "Qwen-480B", "GPT-OSS-120b"
CONTEXT_VARIANT = "Method"   # e.g. "Minimal", "Method", "Class"

# ──────────────────────────────────────────────────────────────────────────────


EXCEPTION_PATTERNS = [
    r'(?:^|\s)([\w\.]+(?:Exception|Error))(?:\s|:|\(|$)',
    r'Caused by:\s+([\w\.]+(?:Exception|Error))',
    r'(?:throws|threw|raised)\s+([\w\.]+(?:Exception|Error))',
]

EXCLUDED = {'Error', 'Exception', 'MojoFailureException', 'MojoExecutionException'}


def parse_bump_errors(raw: str) -> set:
    """
    Parse the exception_types column from BUMP CSV into a set of
    short exception class names (e.g. 'NoClassDefFoundError').
    Pipe-separated values are split and Maven-specific exceptions are excluded.
    """
    if not raw or str(raw).strip() in ('', 'nan'):
        return set()
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        # Extract short class name: java.lang.NoClassDefFoundError → NoClassDefFoundError
        result.add(e.split('.')[-1])
    return result


def parse_log_errors(log_path: Path) -> set:
    """
    Extract all exception/error class names from a test execution log file.
    Returns a set of short class names (e.g. 'NoClassDefFoundError').
    """
    if not log_path.exists():
        return set()
    try:
        content = log_path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return set()
    found = set()
    for pattern in EXCEPTION_PATTERNS:
        for match in re.findall(pattern, content, re.MULTILINE):
            name = match.strip().split('.')[-1]
            if name and name not in EXCLUDED and ('Exception' in name or 'Error' in name):
                if not name.startswith('test'):
                    found.add(name)
    return found


def load_bump(bump_csv: str) -> dict:
    """
    Load BUMP CSV into a dict keyed by instance ID (custom_id).
    Returns per-instance: bump_errors (set), bump_errors_raw (str), failure_category (str).
    """
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            data[row['custom_id']] = {
                'bump_errors':      parse_bump_errors(row.get('exception_types', '')),
                'bump_errors_raw':  row.get('exception_types', ''),
                'failure_category': row.get('failureCategory', ''),
            }
    return data


def load_llm(llm_json: str) -> dict:
    """
    Load LLM test execution results JSON.
    Returns the 'results' dict keyed by instance ID.
    """
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    bump = load_bump(BUMP_CSV)
    llm  = load_llm(LLM_RESULTS_JSON)
    logs = Path(LLM_LOGS_DIR)

    rows = []

    for instance_id, instance_data in llm.items():
        failed_tests = instance_data.get('tests', {}).get('failed', [])

        # Only process instances where LLM detected a BC (has failed tests on v2)
        if not failed_tests:
            continue
        if instance_id not in bump:
            print(f"Warning: {instance_id} not in BUMP CSV — skipping")
            continue

        bump_errors = bump[instance_id]['bump_errors']

        # Collect all exception types from failed test logs
        llm_errors = set()
        for test in failed_tests:
            if test.get('result_type') == 'transplant_issue':
                continue
            test_file = test.get('file', '')
            if not test_file:
                continue
            log_path = logs / f"{instance_id}_{test_file}_breaking_single.log"
            llm_errors.update(parse_log_errors(log_path))

        matched = bump_errors & llm_errors
        missed  = bump_errors - llm_errors
        new     = llm_errors  - bump_errors

        match_rate = round(len(matched) / len(bump_errors) * 100, 1) if bump_errors else 0.0

        rows.append({
            'model':               MODEL_NAME,
            'context_variant':     CONTEXT_VARIANT,
            'instance':            instance_id,
            'bump_bc_failure_category': bump[instance_id]['failure_category'],
            'bump_bc_errors_raw':  bump[instance_id]['bump_errors_raw'],
            'bump_bc_errors':      '|'.join(sorted(bump_errors)),
            'llm_detected_errors': '|'.join(sorted(llm_errors)),
            'matched_errors':      '|'.join(sorted(matched)),
            'missed_errors':       '|'.join(sorted(missed)),
            'new_errors':          '|'.join(sorted(new)),
            'bump_total':          len(bump_errors),
            'llm_total':           len(llm_errors),
            'matched_count':       len(matched),
            'missed_count':        len(missed),
            'new_count':           len(new),
            'match_rate_%':        match_rate,
            'num_failed_tests':    len(failed_tests),
        })

     # Write CSV — append if file exists, write with header if new
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'model', 'context_variant',
        'instance', 'bump_bc_failure_category', 'bump_bc_errors_raw',
        'bump_bc_errors', 'llm_detected_errors', 'matched_errors', 'missed_errors', 'new_errors',
        'bump_total', 'llm_total', 'matched_count', 'missed_count', 'new_count',
        'match_rate_%', 'num_failed_tests',
    ]
    file_exists = Path(OUTPUT_CSV).exists()
    with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()  # only write header if file is new
        writer.writerows(rows)

    print(f"Done. {len(rows)} detected instances written to {OUTPUT_CSV}")
    print(f"Model: {MODEL_NAME} | Context Variant: {CONTEXT_VARIANT}")


if __name__ == "__main__":
    main()

Done. 20 detected instances written to /Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv
Model: Qwen-480B | Context Variant: Method


In [ ]:
import json
import csv
import re
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────

# JSON file with LLM test execution results on breaking version (v2)
LLM_RESULTS_JSON = "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"

# CSV file with BUMP ground-truth breaking change metadata
BUMP_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/BUMPErrorLogs/RQ4_resultsBUMP.csv"

# Directory containing test execution logs for breaking version
LLM_LOGS_DIR = "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/bre/logs"

# Output CSV — one row per instance where LLM detected a BC
OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv"

# Hardcoded experiment metadata — update these before each run
MODEL_NAME      = "Qwen-480B"    # e.g. "GPT-4o", "Qwen-480B", "GPT-OSS-120b"
CONTEXT_VARIANT = "Class"   # e.g. "Minimal", "Method", "Class"

# ──────────────────────────────────────────────────────────────────────────────


EXCEPTION_PATTERNS = [
    r'(?:^|\s)([\w\.]+(?:Exception|Error))(?:\s|:|\(|$)',
    r'Caused by:\s+([\w\.]+(?:Exception|Error))',
    r'(?:throws|threw|raised)\s+([\w\.]+(?:Exception|Error))',
]

EXCLUDED = {'Error', 'Exception', 'MojoFailureException', 'MojoExecutionException'}


def parse_bump_errors(raw: str) -> set:
    """
    Parse the exception_types column from BUMP CSV into a set of
    short exception class names (e.g. 'NoClassDefFoundError').
    Pipe-separated values are split and Maven-specific exceptions are excluded.
    """
    if not raw or str(raw).strip() in ('', 'nan'):
        return set()
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        # Extract short class name: java.lang.NoClassDefFoundError → NoClassDefFoundError
        result.add(e.split('.')[-1])
    return result


def parse_log_errors(log_path: Path) -> set:
    """
    Extract all exception/error class names from a test execution log file.
    Returns a set of short class names (e.g. 'NoClassDefFoundError').
    """
    if not log_path.exists():
        return set()
    try:
        content = log_path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return set()
    found = set()
    for pattern in EXCEPTION_PATTERNS:
        for match in re.findall(pattern, content, re.MULTILINE):
            name = match.strip().split('.')[-1]
            if name and name not in EXCLUDED and ('Exception' in name or 'Error' in name):
                if not name.startswith('test'):
                    found.add(name)
    return found


def load_bump(bump_csv: str) -> dict:
    """
    Load BUMP CSV into a dict keyed by instance ID (custom_id).
    Returns per-instance: bump_errors (set), bump_errors_raw (str), failure_category (str).
    """
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            data[row['custom_id']] = {
                'bump_errors':      parse_bump_errors(row.get('exception_types', '')),
                'bump_errors_raw':  row.get('exception_types', ''),
                'failure_category': row.get('failureCategory', ''),
            }
    return data


def load_llm(llm_json: str) -> dict:
    """
    Load LLM test execution results JSON.
    Returns the 'results' dict keyed by instance ID.
    """
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    bump = load_bump(BUMP_CSV)
    llm  = load_llm(LLM_RESULTS_JSON)
    logs = Path(LLM_LOGS_DIR)

    rows = []

    for instance_id, instance_data in llm.items():
        failed_tests = instance_data.get('tests', {}).get('failed', [])

        # Only process instances where LLM detected a BC (has failed tests on v2)
        if not failed_tests:
            continue
        if instance_id not in bump:
            print(f"Warning: {instance_id} not in BUMP CSV — skipping")
            continue

        bump_errors = bump[instance_id]['bump_errors']

        # Collect all exception types from failed test logs
        llm_errors = set()
        for test in failed_tests:
            if test.get('result_type') == 'transplant_issue':
                continue
            test_file = test.get('file', '')
            if not test_file:
                continue
            log_path = logs / f"{instance_id}_{test_file}_breaking_single.log"
            llm_errors.update(parse_log_errors(log_path))

        matched = bump_errors & llm_errors
        missed  = bump_errors - llm_errors
        new     = llm_errors  - bump_errors

        match_rate = round(len(matched) / len(bump_errors) * 100, 1) if bump_errors else 0.0

        rows.append({
            'model':               MODEL_NAME,
            'context_variant':     CONTEXT_VARIANT,
            'instance':            instance_id,
            'bump_bc_failure_category': bump[instance_id]['failure_category'],
            'bump_bc_errors_raw':  bump[instance_id]['bump_errors_raw'],
            'bump_bc_errors':      '|'.join(sorted(bump_errors)),
            'llm_detected_errors': '|'.join(sorted(llm_errors)),
            'matched_errors':      '|'.join(sorted(matched)),
            'missed_errors':       '|'.join(sorted(missed)),
            'new_errors':          '|'.join(sorted(new)),
            'bump_total':          len(bump_errors),
            'llm_total':           len(llm_errors),
            'matched_count':       len(matched),
            'missed_count':        len(missed),
            'new_count':           len(new),
            'match_rate_%':        match_rate,
            'num_failed_tests':    len(failed_tests),
        })

     # Write CSV — append if file exists, write with header if new
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'model', 'context_variant',
        'instance', 'bump_bc_failure_category', 'bump_bc_errors_raw',
        'bump_bc_errors', 'llm_detected_errors', 'matched_errors', 'missed_errors', 'new_errors',
        'bump_total', 'llm_total', 'matched_count', 'missed_count', 'new_count',
        'match_rate_%', 'num_failed_tests',
    ]
    file_exists = Path(OUTPUT_CSV).exists()
    with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()  # only write header if file is new
        writer.writerows(rows)

    print(f"Done. {len(rows)} detected instances written to {OUTPUT_CSV}")
    print(f"Model: {MODEL_NAME} | Context Variant: {CONTEXT_VARIANT}")


if __name__ == "__main__":
    main()

Done. 20 detected instances written to /Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv
Model: Qwen-480B | Context Variant: Class


GPTOSS Results

In [ ]:
import json
import csv
import re
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────

# JSON file with LLM test execution results on breaking version (v2)
LLM_RESULTS_JSON = "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/transplant_results_breaking_single_module.json"

# CSV file with BUMP ground-truth breaking change metadata
BUMP_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/BUMPErrorLogs/RQ4_resultsBUMP.csv"

# Directory containing test execution logs for breaking version
LLM_LOGS_DIR = "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/bre/logs"

# Output CSV — one row per instance where LLM detected a BC
OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv"

# Hardcoded experiment metadata — update these before each run
MODEL_NAME      = "GPT-OSS-120b"    # e.g. "GPT-4o", "Qwen-480B", "GPT-OSS-120b"
CONTEXT_VARIANT = "Minimal"   # e.g. "Minimal", "Method", "Class"

# ──────────────────────────────────────────────────────────────────────────────


EXCEPTION_PATTERNS = [
    r'(?:^|\s)([\w\.]+(?:Exception|Error))(?:\s|:|\(|$)',
    r'Caused by:\s+([\w\.]+(?:Exception|Error))',
    r'(?:throws|threw|raised)\s+([\w\.]+(?:Exception|Error))',
]

EXCLUDED = {'Error', 'Exception', 'MojoFailureException', 'MojoExecutionException'}


def parse_bump_errors(raw: str) -> set:
    """
    Parse the exception_types column from BUMP CSV into a set of
    short exception class names (e.g. 'NoClassDefFoundError').
    Pipe-separated values are split and Maven-specific exceptions are excluded.
    """
    if not raw or str(raw).strip() in ('', 'nan'):
        return set()
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        # Extract short class name: java.lang.NoClassDefFoundError → NoClassDefFoundError
        result.add(e.split('.')[-1])
    return result


def parse_log_errors(log_path: Path) -> set:
    """
    Extract all exception/error class names from a test execution log file.
    Returns a set of short class names (e.g. 'NoClassDefFoundError').
    """
    if not log_path.exists():
        return set()
    try:
        content = log_path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return set()
    found = set()
    for pattern in EXCEPTION_PATTERNS:
        for match in re.findall(pattern, content, re.MULTILINE):
            name = match.strip().split('.')[-1]
            if name and name not in EXCLUDED and ('Exception' in name or 'Error' in name):
                if not name.startswith('test'):
                    found.add(name)
    return found


def load_bump(bump_csv: str) -> dict:
    """
    Load BUMP CSV into a dict keyed by instance ID (custom_id).
    Returns per-instance: bump_errors (set), bump_errors_raw (str), failure_category (str).
    """
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            data[row['custom_id']] = {
                'bump_errors':      parse_bump_errors(row.get('exception_types', '')),
                'bump_errors_raw':  row.get('exception_types', ''),
                'failure_category': row.get('failureCategory', ''),
            }
    return data


def load_llm(llm_json: str) -> dict:
    """
    Load LLM test execution results JSON.
    Returns the 'results' dict keyed by instance ID.
    """
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    bump = load_bump(BUMP_CSV)
    llm  = load_llm(LLM_RESULTS_JSON)
    logs = Path(LLM_LOGS_DIR)

    rows = []

    for instance_id, instance_data in llm.items():
        failed_tests = instance_data.get('tests', {}).get('failed', [])

        # Only process instances where LLM detected a BC (has failed tests on v2)
        if not failed_tests:
            continue
        if instance_id not in bump:
            print(f"Warning: {instance_id} not in BUMP CSV — skipping")
            continue

        bump_errors = bump[instance_id]['bump_errors']

        # Collect all exception types from failed test logs
        llm_errors = set()
        for test in failed_tests:
            if test.get('result_type') == 'transplant_issue':
                continue
            test_file = test.get('file', '')
            if not test_file:
                continue
            log_path = logs / f"{instance_id}_{test_file}_breaking_single.log"
            llm_errors.update(parse_log_errors(log_path))

        matched = bump_errors & llm_errors
        missed  = bump_errors - llm_errors
        new     = llm_errors  - bump_errors

        match_rate = round(len(matched) / len(bump_errors) * 100, 1) if bump_errors else 0.0

        rows.append({
            'model':               MODEL_NAME,
            'context_variant':     CONTEXT_VARIANT,
            'instance':            instance_id,
            'bump_bc_failure_category': bump[instance_id]['failure_category'],
            'bump_bc_errors_raw':  bump[instance_id]['bump_errors_raw'],
            'bump_bc_errors':      '|'.join(sorted(bump_errors)),
            'llm_detected_errors': '|'.join(sorted(llm_errors)),
            'matched_errors':      '|'.join(sorted(matched)),
            'missed_errors':       '|'.join(sorted(missed)),
            'new_errors':          '|'.join(sorted(new)),
            'bump_total':          len(bump_errors),
            'llm_total':           len(llm_errors),
            'matched_count':       len(matched),
            'missed_count':        len(missed),
            'new_count':           len(new),
            'match_rate_%':        match_rate,
            'num_failed_tests':    len(failed_tests),
        })

     # Write CSV — append if file exists, write with header if new
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'model', 'context_variant',
        'instance', 'bump_bc_failure_category', 'bump_bc_errors_raw',
        'bump_bc_errors', 'llm_detected_errors', 'matched_errors', 'missed_errors', 'new_errors',
        'bump_total', 'llm_total', 'matched_count', 'missed_count', 'new_count',
        'match_rate_%', 'num_failed_tests',
    ]
    file_exists = Path(OUTPUT_CSV).exists()
    with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()  # only write header if file is new
        writer.writerows(rows)

    print(f"Done. {len(rows)} detected instances written to {OUTPUT_CSV}")
    print(f"Model: {MODEL_NAME} | Context Variant: {CONTEXT_VARIANT}")


if __name__ == "__main__":
    main()

Done. 4 detected instances written to /Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv
Model: GPT-OSS-120b | Context Variant: Minimal


In [ ]:
import json
import csv
import re
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────

# JSON file with LLM test execution results on breaking version (v2)
LLM_RESULTS_JSON = "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/transplant_results_breaking_single_module.json"

# CSV file with BUMP ground-truth breaking change metadata
BUMP_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/BUMPErrorLogs/RQ4_resultsBUMP.csv"

# Directory containing test execution logs for breaking version
LLM_LOGS_DIR = "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/bre/logs"

# Output CSV — one row per instance where LLM detected a BC
OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv"

# Hardcoded experiment metadata — update these before each run
MODEL_NAME      = "GPT-OSS-120b"    # e.g. "GPT-4o", "Qwen-480B", "GPT-OSS-120b"
CONTEXT_VARIANT = "Method"   # e.g. "Minimal", "Method", "Class"

# ──────────────────────────────────────────────────────────────────────────────


EXCEPTION_PATTERNS = [
    r'(?:^|\s)([\w\.]+(?:Exception|Error))(?:\s|:|\(|$)',
    r'Caused by:\s+([\w\.]+(?:Exception|Error))',
    r'(?:throws|threw|raised)\s+([\w\.]+(?:Exception|Error))',
]

EXCLUDED = {'Error', 'Exception', 'MojoFailureException', 'MojoExecutionException'}


def parse_bump_errors(raw: str) -> set:
    """
    Parse the exception_types column from BUMP CSV into a set of
    short exception class names (e.g. 'NoClassDefFoundError').
    Pipe-separated values are split and Maven-specific exceptions are excluded.
    """
    if not raw or str(raw).strip() in ('', 'nan'):
        return set()
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        # Extract short class name: java.lang.NoClassDefFoundError → NoClassDefFoundError
        result.add(e.split('.')[-1])
    return result


def parse_log_errors(log_path: Path) -> set:
    """
    Extract all exception/error class names from a test execution log file.
    Returns a set of short class names (e.g. 'NoClassDefFoundError').
    """
    if not log_path.exists():
        return set()
    try:
        content = log_path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return set()
    found = set()
    for pattern in EXCEPTION_PATTERNS:
        for match in re.findall(pattern, content, re.MULTILINE):
            name = match.strip().split('.')[-1]
            if name and name not in EXCLUDED and ('Exception' in name or 'Error' in name):
                if not name.startswith('test'):
                    found.add(name)
    return found


def load_bump(bump_csv: str) -> dict:
    """
    Load BUMP CSV into a dict keyed by instance ID (custom_id).
    Returns per-instance: bump_errors (set), bump_errors_raw (str), failure_category (str).
    """
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            data[row['custom_id']] = {
                'bump_errors':      parse_bump_errors(row.get('exception_types', '')),
                'bump_errors_raw':  row.get('exception_types', ''),
                'failure_category': row.get('failureCategory', ''),
            }
    return data


def load_llm(llm_json: str) -> dict:
    """
    Load LLM test execution results JSON.
    Returns the 'results' dict keyed by instance ID.
    """
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    bump = load_bump(BUMP_CSV)
    llm  = load_llm(LLM_RESULTS_JSON)
    logs = Path(LLM_LOGS_DIR)

    rows = []

    for instance_id, instance_data in llm.items():
        failed_tests = instance_data.get('tests', {}).get('failed', [])

        # Only process instances where LLM detected a BC (has failed tests on v2)
        if not failed_tests:
            continue
        if instance_id not in bump:
            print(f"Warning: {instance_id} not in BUMP CSV — skipping")
            continue

        bump_errors = bump[instance_id]['bump_errors']

        # Collect all exception types from failed test logs
        llm_errors = set()
        for test in failed_tests:
            if test.get('result_type') == 'transplant_issue':
                continue
            test_file = test.get('file', '')
            if not test_file:
                continue
            log_path = logs / f"{instance_id}_{test_file}_breaking_single.log"
            llm_errors.update(parse_log_errors(log_path))

        matched = bump_errors & llm_errors
        missed  = bump_errors - llm_errors
        new     = llm_errors  - bump_errors

        match_rate = round(len(matched) / len(bump_errors) * 100, 1) if bump_errors else 0.0

        rows.append({
            'model':               MODEL_NAME,
            'context_variant':     CONTEXT_VARIANT,
            'instance':            instance_id,
            'bump_bc_failure_category': bump[instance_id]['failure_category'],
            'bump_bc_errors_raw':  bump[instance_id]['bump_errors_raw'],
            'bump_bc_errors':      '|'.join(sorted(bump_errors)),
            'llm_detected_errors': '|'.join(sorted(llm_errors)),
            'matched_errors':      '|'.join(sorted(matched)),
            'missed_errors':       '|'.join(sorted(missed)),
            'new_errors':          '|'.join(sorted(new)),
            'bump_total':          len(bump_errors),
            'llm_total':           len(llm_errors),
            'matched_count':       len(matched),
            'missed_count':        len(missed),
            'new_count':           len(new),
            'match_rate_%':        match_rate,
            'num_failed_tests':    len(failed_tests),
        })

     # Write CSV — append if file exists, write with header if new
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'model', 'context_variant',
        'instance', 'bump_bc_failure_category', 'bump_bc_errors_raw',
        'bump_bc_errors', 'llm_detected_errors', 'matched_errors', 'missed_errors', 'new_errors',
        'bump_total', 'llm_total', 'matched_count', 'missed_count', 'new_count',
        'match_rate_%', 'num_failed_tests',
    ]
    file_exists = Path(OUTPUT_CSV).exists()
    with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()  # only write header if file is new
        writer.writerows(rows)

    print(f"Done. {len(rows)} detected instances written to {OUTPUT_CSV}")
    print(f"Model: {MODEL_NAME} | Context Variant: {CONTEXT_VARIANT}")


if __name__ == "__main__":
    main()

Done. 8 detected instances written to /Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv
Model: GPT-OSS-120b | Context Variant: Method


In [ ]:
import json
import csv
import re
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────

# JSON file with LLM test execution results on breaking version (v2)
LLM_RESULTS_JSON = "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/transplant_results_breaking_single_module.json"

# CSV file with BUMP ground-truth breaking change metadata
BUMP_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/BUMPErrorLogs/RQ4_resultsBUMP.csv"

# Directory containing test execution logs for breaking version
LLM_LOGS_DIR = "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/bre/logs"

# Output CSV — one row per instance where LLM detected a BC
OUTPUT_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv"

# Hardcoded experiment metadata — update these before each run
MODEL_NAME      = "GPT-OSS-120b"    # e.g. "GPT-4o", "Qwen-480B", "GPT-OSS-120b"
CONTEXT_VARIANT = "Class"   # e.g. "Minimal", "Method", "Class"

# ──────────────────────────────────────────────────────────────────────────────


EXCEPTION_PATTERNS = [
    r'(?:^|\s)([\w\.]+(?:Exception|Error))(?:\s|:|\(|$)',
    r'Caused by:\s+([\w\.]+(?:Exception|Error))',
    r'(?:throws|threw|raised)\s+([\w\.]+(?:Exception|Error))',
]

EXCLUDED = {'Error', 'Exception', 'MojoFailureException', 'MojoExecutionException'}


def parse_bump_errors(raw: str) -> set:
    """
    Parse the exception_types column from BUMP CSV into a set of
    short exception class names (e.g. 'NoClassDefFoundError').
    Pipe-separated values are split and Maven-specific exceptions are excluded.
    """
    if not raw or str(raw).strip() in ('', 'nan'):
        return set()
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or 'MojoFailureException' in e or 'MojoExecutionException' in e:
            continue
        # Extract short class name: java.lang.NoClassDefFoundError → NoClassDefFoundError
        result.add(e.split('.')[-1])
    return result


def parse_log_errors(log_path: Path) -> set:
    """
    Extract all exception/error class names from a test execution log file.
    Returns a set of short class names (e.g. 'NoClassDefFoundError').
    """
    if not log_path.exists():
        return set()
    try:
        content = log_path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        return set()
    found = set()
    for pattern in EXCEPTION_PATTERNS:
        for match in re.findall(pattern, content, re.MULTILINE):
            name = match.strip().split('.')[-1]
            if name and name not in EXCLUDED and ('Exception' in name or 'Error' in name):
                if not name.startswith('test'):
                    found.add(name)
    return found


def load_bump(bump_csv: str) -> dict:
    """
    Load BUMP CSV into a dict keyed by instance ID (custom_id).
    Returns per-instance: bump_errors (set), bump_errors_raw (str), failure_category (str).
    """
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            data[row['custom_id']] = {
                'bump_errors':      parse_bump_errors(row.get('exception_types', '')),
                'bump_errors_raw':  row.get('exception_types', ''),
                'failure_category': row.get('failureCategory', ''),
            }
    return data


def load_llm(llm_json: str) -> dict:
    """
    Load LLM test execution results JSON.
    Returns the 'results' dict keyed by instance ID.
    """
    with open(llm_json) as f:
        data = json.load(f)
    return data.get('results', data)


def main():
    bump = load_bump(BUMP_CSV)
    llm  = load_llm(LLM_RESULTS_JSON)
    logs = Path(LLM_LOGS_DIR)

    rows = []

    for instance_id, instance_data in llm.items():
        failed_tests = instance_data.get('tests', {}).get('failed', [])

        # Only process instances where LLM detected a BC (has failed tests on v2)
        if not failed_tests:
            continue
        if instance_id not in bump:
            print(f"Warning: {instance_id} not in BUMP CSV — skipping")
            continue

        bump_errors = bump[instance_id]['bump_errors']

        # Collect all exception types from failed test logs
        llm_errors = set()
        for test in failed_tests:
            if test.get('result_type') == 'transplant_issue':
                continue
            test_file = test.get('file', '')
            if not test_file:
                continue
            log_path = logs / f"{instance_id}_{test_file}_breaking_single.log"
            llm_errors.update(parse_log_errors(log_path))

        matched = bump_errors & llm_errors
        missed  = bump_errors - llm_errors
        new     = llm_errors  - bump_errors

        match_rate = round(len(matched) / len(bump_errors) * 100, 1) if bump_errors else 0.0

        rows.append({
            'model':               MODEL_NAME,
            'context_variant':     CONTEXT_VARIANT,
            'instance':            instance_id,
            'bump_bc_failure_category': bump[instance_id]['failure_category'],
            'bump_bc_errors_raw':  bump[instance_id]['bump_errors_raw'],
            'bump_bc_errors':      '|'.join(sorted(bump_errors)),
            'llm_detected_errors': '|'.join(sorted(llm_errors)),
            'matched_errors':      '|'.join(sorted(matched)),
            'missed_errors':       '|'.join(sorted(missed)),
            'new_errors':          '|'.join(sorted(new)),
            'bump_total':          len(bump_errors),
            'llm_total':           len(llm_errors),
            'matched_count':       len(matched),
            'missed_count':        len(missed),
            'new_count':           len(new),
            'match_rate_%':        match_rate,
            'num_failed_tests':    len(failed_tests),
        })

     # Write CSV — append if file exists, write with header if new
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    fieldnames = [
        'model', 'context_variant',
        'instance', 'bump_bc_failure_category', 'bump_bc_errors_raw',
        'bump_bc_errors', 'llm_detected_errors', 'matched_errors', 'missed_errors', 'new_errors',
        'bump_total', 'llm_total', 'matched_count', 'missed_count', 'new_count',
        'match_rate_%', 'num_failed_tests',
    ]
    file_exists = Path(OUTPUT_CSV).exists()
    with open(OUTPUT_CSV, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()  # only write header if file is new
        writer.writerows(rows)

    print(f"Done. {len(rows)} detected instances written to {OUTPUT_CSV}")
    print(f"Model: {MODEL_NAME} | Context Variant: {CONTEXT_VARIANT}")


if __name__ == "__main__":
    main()

Done. 10 detected instances written to /Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv
Model: GPT-OSS-120b | Context Variant: Class


Error Type Coverage Visualization for paper

In [2]:
"""
analysis1_error_type_coverage.py
=================================
Generates Analysis 1 visualization for RQ3:
Error type coverage — for each BUMP error type, how many instances
did LLM-generated tests detect vs total instances containing that error type,
broken down by model and context variant (9 combinations).

INPUT FILES:
------------
1. DETECTED_CSV — output from detected_bc_errortype_coverage.py
   One row per detected instance per model-variant combination.
   Key columns: model, context_variant, instance, matched_errors, bump_bc_errors

2. BUMP_CSV — ground truth for all 89 BUMP instances
   Key columns: custom_id, exception_types

OUTPUT:
-------
1. OUTPUT_TABLE_CSV  — per error type per model-variant: total, detected, detection_rate_%
2. OUTPUT_PLOT_PNG   — heatmap + grouped bar chart side by side
"""

import csv
import re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns


# ─── CONFIG ───────────────────────────────────────────────────────────────────
DETECTED_CSV     = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv"
BUMP_CSV         = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes.csv"
OUTPUT_TABLE_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/analysis1_error_type_coverage_table.csv"

# Error types to focus on — matches BUMP Table V categories
# Edit this list to match the exact short names in your data
FOCUS_ERROR_TYPES = [
    'NoClassDefFoundError',
    'IllegalStateException',
    'ClassCastException',
    'UnsupportedClassVersionError',
    'AssertionFailedError',
    'AssertionError',
    'NoSuchMethodError',
]

# Model-variant display order
MODEL_ORDER   = ['GPT-4o', 'Qwen-480B', 'GPT-OSS-120b']
VARIANT_ORDER = ['Minimal', 'Method', 'Class']
# ──────────────────────────────────────────────────────────────────────────────


MAVEN_EXCLUDED = {'MojoFailureException', 'MojoExecutionException'}


def parse_errors(raw: str) -> set:
    """Parse pipe-separated error types, return set of short class names."""
    if not raw or str(raw).strip() in ('', 'nan'):
        return set()
    result = set()
    for e in str(raw).split('|'):
        e = e.strip()
        if not e or any(ex in e for ex in MAVEN_EXCLUDED):
            continue
        result.add(e.split('.')[-1])
    return result


def load_bump_error_types(bump_csv: str) -> dict:
    """
    Load BUMP CSV.
    Returns dict: { instance_id -> set of error types }
    """
    data = {}
    with open(bump_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            data[row['custom_id']] = parse_errors(row.get('exception_types', ''))
    return data


def load_detected(detected_csv: str) -> list:
    """
    Load detected BC coverage CSV.
    Returns list of dicts with model, context_variant, instance, matched_errors.
    """
    rows = []
    with open(detected_csv, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            rows.append({
                'model':           row['model'],
                'context_variant': row['context_variant'],
                'instance':        row['instance'],
                'matched_errors':  parse_errors(row.get('matched_errors', '')),
                'bump_bc_errors':  parse_errors(row.get('bump_bc_errors', '')),
            })
    return rows


def build_coverage_table(bump_data: dict, detected_rows: list) -> pd.DataFrame:
    """
    For each model-variant combination and each error type:
    - total: number of BUMP instances containing that error type
    - detected: number of those instances where LLM matched that error type
    - detection_rate_%
    """
    # Build lookup: (model, variant, instance) -> matched_errors
    detected_lookup = defaultdict(set)
    for row in detected_rows:
        key = (row['model'], row['context_variant'], row['instance'])
        detected_lookup[key].update(row['matched_errors'])

    records = []
    for model in MODEL_ORDER:
        for variant in VARIANT_ORDER:
            for error_type in FOCUS_ERROR_TYPES:
                # Total BUMP instances with this error type
                total_instances = [
                    inst for inst, errors in bump_data.items()
                    if error_type in errors
                ]
                total = len(total_instances)

                # Detected: LLM matched this error type for this instance
                detected = sum(
                    1 for inst in total_instances
                    if error_type in detected_lookup.get((model, variant, inst), set())
                )

                rate = round(detected / total * 100, 1) if total > 0 else 0.0

                records.append({
                    'model':            model,
                    'context_variant':  variant,
                    'model_variant':    f"{model}\n{variant}",
                    'error_type':       error_type,
                    'total':            total,
                    'detected':         detected,
                    'detection_rate_%': rate,
                })

    return pd.DataFrame(records)


def plot_coverage(df: pd.DataFrame, output_path: str):
    """
    Generate side-by-side heatmap and grouped bar chart.
    """
    fig = plt.figure(figsize=(20, 8))
    gs  = gridspec.GridSpec(1, 2, width_ratios=[1.2, 1.8], wspace=0.4)

    # ── Heatmap ───────────────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0])

    # Pivot: rows = model-variant, cols = error type
    pivot = df.pivot_table(
        index='model_variant',
        columns='error_type',
        values='detection_rate_%'
    )
    # Reorder rows
    row_order = [f"{m}\n{v}" for m in MODEL_ORDER for v in VARIANT_ORDER]
    pivot = pivot.reindex([r for r in row_order if r in pivot.index])
    pivot = pivot[FOCUS_ERROR_TYPES]  # reorder columns

    # Annotation: "detected/total\n rate%"
    annot_df = df.pivot_table(
        index='model_variant', columns='error_type',
        values='detected', aggfunc='first'
    ).reindex(pivot.index)[FOCUS_ERROR_TYPES]
    total_df = df.pivot_table(
        index='model_variant', columns='error_type',
        values='total', aggfunc='first'
    ).reindex(pivot.index)[FOCUS_ERROR_TYPES]

    annot = annot_df.astype(str) + '/' + total_df.astype(str) + '\n' + pivot.round(1).astype(str) + '%'

    sns.heatmap(
        pivot,
        ax=ax1,
        annot=annot,
        fmt='',
        cmap='RdYlGn',
        vmin=0, vmax=100,
        linewidths=0.5,
        cbar_kws={'label': 'Detection Rate (%)'},
        annot_kws={'size': 8},
    )
    ax1.set_title('Detection Rate by Error Type\nand Model-Variant', fontsize=12, fontweight='bold')
    ax1.set_xlabel('BUMP Error Type', fontsize=10)
    ax1.set_ylabel('Model / Context Variant', fontsize=10)
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=35, ha='right', fontsize=8)
    ax1.set_yticklabels(ax1.get_yticklabels(), rotation=0, fontsize=8)

    # ── Grouped Bar Chart ─────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1])

    x      = np.arange(len(FOCUS_ERROR_TYPES))
    combos = [f"{m}-{v}" for m in MODEL_ORDER for v in VARIANT_ORDER]
    colors = plt.cm.tab10(np.linspace(0, 0.9, len(combos)))
    width  = 0.08

    for i, (model, variant) in enumerate(
        [(m, v) for m in MODEL_ORDER for v in VARIANT_ORDER]
    ):
        subset = df[(df['model'] == model) & (df['context_variant'] == variant)]
        rates  = [
            subset[subset['error_type'] == et]['detection_rate_%'].values[0]
            if len(subset[subset['error_type'] == et]) > 0 else 0
            for et in FOCUS_ERROR_TYPES
        ]
        offset = (i - len(combos) / 2) * width + width / 2
        ax2.bar(x + offset, rates, width, label=f"{model}-{variant}", color=colors[i])

    ax2.set_xticks(x)
    ax2.set_xticklabels(FOCUS_ERROR_TYPES, rotation=35, ha='right', fontsize=8)
    ax2.set_ylabel('Detection Rate (%)', fontsize=10)
    ax2.set_title('Detection Rate by Error Type\nand Model-Variant (Grouped Bar)', fontsize=12, fontweight='bold')
    ax2.set_ylim(0, 110)
    ax2.legend(fontsize=7, ncol=2, loc='upper right')
    ax2.axhline(y=100, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

    plt.suptitle('RQ3 Analysis 1: BUMP Error Type Detection Coverage', fontsize=14, fontweight='bold', y=1.01)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Plot saved to: {output_path}")


def main():
    print("Loading data...")
    bump_data     = load_bump_error_types(BUMP_CSV)
    detected_rows = load_detected(DETECTED_CSV)
    print(f"  BUMP instances loaded:    {len(bump_data)}")
    print(f"  Detected rows loaded:     {len(detected_rows)}")

    print("Building coverage table...")
    df = build_coverage_table(bump_data, detected_rows)

    # Save table CSV
    Path(OUTPUT_TABLE_CSV).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_TABLE_CSV, index=False)
    print(f"Table saved to: {OUTPUT_TABLE_CSV}")

    # Print quick summary
    print("\nDetection rate summary (averaged across model-variants):")
    summary = df.groupby('error_type')['detection_rate_%'].mean().round(1).sort_values(ascending=False)
    for et, rate in summary.items():
        total = df[df['error_type'] == et]['total'].iloc[0]
        print(f"  {et:35s}: {rate:5.1f}%  (BUMP total: {total})")

    print("\nGenerating plots...")
    plot_coverage(df, OUTPUT_PLOT_PNG)
    print("Done.")


if __name__ == "__main__":
    main()

Loading data...
  BUMP instances loaded:    89
  Detected rows loaded:     137
Building coverage table...
Table saved to: /Volumes/Rachna-HD/RQResultsForPaper/RQ3/analysis1_error_type_coverage_table.csv

Detection rate summary (averaged across model-variants):
  NoSuchMethodError                  :  48.6%  (BUMP total: 8)
  NoClassDefFoundError               :  30.1%  (BUMP total: 35)
  AssertionError                     :   0.0%  (BUMP total: 10)
  AssertionFailedError               :   0.0%  (BUMP total: 7)
  ClassCastException                 :   0.0%  (BUMP total: 10)
  IllegalStateException              :   0.0%  (BUMP total: 1)
  UnsupportedClassVersionError       :   0.0%  (BUMP total: 8)

Generating plots...
Plot saved to: /Volumes/Rachna-HD/RQResultsForPaper/RQ3/analysis1_error_type_coverage.png
Done.


In [10]:
"""
analysis1_error_type_coverage.py
=================================
Pipeline:
  1. BUMP CSV      → reads normalized_error_category column directly
                     (pre-computed by add_error_category_to_bump.py)
                     bump_total = # instances per category, restricted to
                     instances present in detected CSV
  2. Detected CSV  → per (model, context_variant, custom_id):
                       llm_detected_errors : what LLM threw  → normalized
                       matched_errors      : intersection     → normalized
  3. For each (model, context_variant, error_category):
       bump_total   = # BUMP instances with this category
       llm_detected = # instances where LLM detected this category
       in_common    = # instances where both BUMP and LLM share this category
       rate         = in_common / bump_total * 100
"""

import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import defaultdict
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────
BUMP_CSV         = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes_with_categories.csv"
DETECTED_CSV     = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv"
OUTPUT_TABLE_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/analysis1_error_type_coverage_table.csv"
OUTPUT_PLOT_PNG  = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/analysis1_error_type_coverage.png"

MODEL_ORDER   = ['GPT-4o', 'Qwen-480B', 'GPT-OSS-120b']
VARIANT_ORDER = ['Minimal', 'Method', 'Class']

MAVEN_EXCLUDED = {'MojoFailureException', 'MojoExecutionException'}
OTHER = 'Other'
# ──────────────────────────────────────────────────────────────────────────────


def parse_pipe_raw(val) -> set[str]:
    """Pipe-separated raw error string → set of short class names, Maven excluded."""
    if not val or str(val).strip() in ('', 'nan'):
        return set()
    return {
        e.strip().split('.')[-1]
        for e in str(val).split('|')
        if e.strip() and not any(ex in e for ex in MAVEN_EXCLUDED)
    }


def parse_pipe_cats(val) -> set[str]:
    """Pipe-separated already-normalized categories → set of category strings."""
    if not val or str(val).strip() in ('', 'nan'):
        return set()
    return {c.strip() for c in str(val).split('|') if c.strip()}


# ─── STEP 1: Load detected CSV ────────────────────────────────────────────────

def load_detected(path: str) -> tuple[pd.DataFrame, set[str]]:
    df = pd.read_csv(path, sep=None, engine='python', dtype=str, keep_default_na=False)
    df['custom_id'] = df['custom_id'].astype(str).str.strip()

    # LLM errors are raw — parse short names, Maven excluded
    df['_llm_cats']     = df['llm_detected_errors'].apply(parse_pipe_raw)
    df['_matched_cats'] = df['matched_errors'].apply(parse_pipe_raw)

    detected_ids = set(df['custom_id'].unique())
    print(f"  Detected CSV: {len(df)} rows | "
          f"{len(detected_ids)} unique instances | "
          f"{df['model'].nunique()} models | "
          f"{df['context_variant'].nunique()} variants")
    return df, detected_ids


# ─── STEP 2: Load BUMP CSV ────────────────────────────────────────────────────

def load_bump(bump_csv: str, detected_ids: set[str]) -> tuple[dict[str, set[str]], list[str]]:
    """
    Reads normalized_error_category directly — no re-parsing of exception_types.
    Returns:
      bump_cats  : { custom_id -> set of normalized categories }
      all_cats   : ordered list of categories present in BUMP data
    """
    df = pd.read_csv(bump_csv, sep=None, engine='python', dtype=str, keep_default_na=False)
    df['custom_id'] = df['custom_id'].astype(str).str.strip()

    bump_cats: dict[str, set[str]] = {}
    counts: dict[str, int] = defaultdict(int)

    for _, row in df.iterrows():
        cid = row['custom_id']
        if cid not in detected_ids:
            continue
        cats = parse_pipe_cats(row.get('normalized_error_category', ''))
        bump_cats[cid] = cats
        for c in cats:
            counts[c] += 1

    # Order by frequency descending, Other always last
    all_cats = sorted(
        (c for c in counts if c != OTHER), key=lambda c: -counts[c]
    ) + ([OTHER] if OTHER in counts else [])

    print(f"  BUMP CSV: {len(bump_cats)} instances matched to detected CSV")
    print(f"  Categories present ({len(all_cats)}):")
    for c in all_cats:
        print(f"    {c:45s}  ({counts[c]} instances)")

    return bump_cats, all_cats


# ─── STEP 3: Build coverage table ────────────────────────────────────────────

def build_table(
    bump_cats: dict[str, set[str]],
    all_cats: list[str],
    df: pd.DataFrame,
) -> pd.DataFrame:
    bump_total_by_cat = {
        cat: sum(1 for cats in bump_cats.values() if cat in cats)
        for cat in all_cats
    }

    records = []
    for model in MODEL_ORDER:
        for variant in VARIANT_ORDER:
            sub = df[(df['model'] == model) & (df['context_variant'] == variant)]

            for cat in all_cats:
                bump_total   = bump_total_by_cat[cat]
                llm_detected = int(sub['_llm_cats'].apply(lambda s: cat in s).sum())
                in_common    = int(sub['_matched_cats'].apply(lambda s: cat in s).sum())
                rate         = round(in_common / bump_total * 100, 1) if bump_total > 0 else 0.0

                records.append({
                    'model':            model,
                    'context_variant':  variant,
                    'model_variant':    f"{model}\n{variant}",
                    'error_category':   cat,
                    'bump_total':       bump_total,
                    'llm_detected':     llm_detected,
                    'in_common':        in_common,
                    'detection_rate_%': rate,
                })

    return pd.DataFrame(records)


# ─── STEP 4: LaTeX tables ─────────────────────────────────────────────────────

def print_latex(df: pd.DataFrame, all_cats: list[str]):
    col_order = [(m, v) for m in MODEL_ORDER for v in VARIANT_ORDER]
    cmidrules = "".join(
        f" \\cmidrule(lr){{{3 + i*3}-{3 + i*3 + 2}}}"
        for i in range(len(MODEL_ORDER))
    )

    def _header(caption, label):
        print(r"\begin{table*}[htbp]")
        print(r"\centering")
        print(f"\\caption{{{caption}}}")
        print(f"\\label{{{label}}}")
        print(r"\resizebox{\textwidth}{!}{%")
        print(r"\begin{tabular}{l r" + " ccc" * len(MODEL_ORDER) + "}")
        print(r"\toprule")
        print("& " + "".join(
            f" & \\multicolumn{{3}}{{c}}{{\\textbf{{{m}}}}}" for m in MODEL_ORDER
        ) + r" \\")
        print(cmidrules + r" \\")
        print(r"\textbf{Error Category} & \textbf{BUMP $n$}" + "".join(
            f" & \\textit{{{v}}}" for _, v in col_order
        ) + r" \\")
        print(r"\midrule")

    def _footer():
        print(r"\bottomrule")
        print(r"\end{tabular}%")
        print(r"}")
        print(r"\end{table*}")

    # Table A: detection rates
    print("\n% ── Table A: Detection Rate % (in_common / bump_total) ────────────")
    _header(r"Error Category Detection Rates (\%) by Model and Context Variant",
            "tab:detection_rates")
    for cat in all_cats:
        bump_n = df[df['error_category'] == cat]['bump_total'].iloc[0]
        vals = [
            f"{df[(df['model']==m) & (df['context_variant']==v) & (df['error_category']==cat)]['detection_rate_%'].values[0]:.1f}"
            for m, v in col_order
        ]
        print(f"\\texttt{{{cat}}} & {bump_n} & " + " & ".join(vals) + r" \\")
    _footer()

    # Table B: raw counts LLM / in-common
    print("\n% ── Table B: Raw Counts (LLM detected / in common) ─────────────────")
    _header(r"Error Category Counts: BUMP Total, LLM Detected, In Common",
            "tab:error_counts")
    for cat in all_cats:
        bump_n = df[df['error_category'] == cat]['bump_total'].iloc[0]
        vals = []
        for m, v in col_order:
            r = df[(df['model']==m) & (df['context_variant']==v) & (df['error_category']==cat)]
            vals.append(f"{r['llm_detected'].values[0]}/{r['in_common'].values[0]}")
        print(f"\\texttt{{{cat}}} & {bump_n} & " + " & ".join(vals) + r" \\")
    _footer()


# ─── STEP 5: Plot ─────────────────────────────────────────────────────────────

def plot_coverage(df: pd.DataFrame, all_cats: list[str], output_path: str):
    fig = plt.figure(figsize=(22, 9))
    gs  = gridspec.GridSpec(1, 2, width_ratios=[1.3, 1.7], wspace=0.4)

    ax1 = fig.add_subplot(gs[0])
    row_order = [f"{m}\n{v}" for m in MODEL_ORDER for v in VARIANT_ORDER]
    pivot = (
        df.pivot_table(index='model_variant', columns='error_category', values='detection_rate_%')
          .reindex([r for r in row_order if r in df['model_variant'].values])
          .reindex(columns=all_cats)
    )
    det   = df.pivot_table(index='model_variant', columns='error_category', values='in_common',  aggfunc='first').reindex(pivot.index).reindex(columns=all_cats)
    total = df.pivot_table(index='model_variant', columns='error_category', values='bump_total', aggfunc='first').reindex(pivot.index).reindex(columns=all_cats)
    annot = det.astype(str) + '/' + total.astype(str) + '\n' + pivot.round(1).astype(str) + '%'

    sns.heatmap(pivot, ax=ax1, annot=annot, fmt='', cmap='RdYlGn',
                vmin=0, vmax=100, linewidths=0.5,
                cbar_kws={'label': 'Detection Rate (%)'},
                annot_kws={'size': 7})
    ax1.set_title('Detection Rate (in common / BUMP total)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Error Category', fontsize=10)
    ax1.set_ylabel('Model / Context Variant', fontsize=10)
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=40, ha='right', fontsize=8)
    ax1.set_yticklabels(ax1.get_yticklabels(), rotation=0, fontsize=8)

    ax2 = fig.add_subplot(gs[1])
    x      = np.arange(len(all_cats))
    combos = [(m, v) for m in MODEL_ORDER for v in VARIANT_ORDER]
    colors = plt.cm.tab10(np.linspace(0, 0.9, len(combos)))
    width  = 0.08
    for i, (model, variant) in enumerate(combos):
        sub   = df[(df['model'] == model) & (df['context_variant'] == variant)]
        rates = [
            sub[sub['error_category'] == c]['detection_rate_%'].values[0]
            if len(sub[sub['error_category'] == c]) > 0 else 0
            for c in all_cats
        ]
        ax2.bar(x + (i - len(combos)/2) * width + width/2, rates, width,
                label=f"{model}-{variant}", color=colors[i])

    ax2.set_xticks(x)
    ax2.set_xticklabels(all_cats, rotation=40, ha='right', fontsize=8)
    ax2.set_ylabel('Detection Rate (%)', fontsize=10)
    ax2.set_title('Detection Rate by Model-Variant', fontsize=12, fontweight='bold')
    ax2.set_ylim(0, 110)
    ax2.legend(fontsize=7, ncol=2, loc='upper right')
    ax2.axhline(y=100, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

    plt.suptitle('RQ3: BUMP Error Category Detection Coverage', fontsize=14, fontweight='bold', y=1.01)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Plot saved → {output_path}")


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    print("\nStep 1: Loading detected CSV...")
    df, detected_ids = load_detected(DETECTED_CSV)

    print("\nStep 2: Loading BUMP CSV (using normalized_error_category)...")
    bump_cats, all_cats = load_bump(BUMP_CSV, detected_ids)

    print("\nStep 3: Building coverage table...")
    table = build_table(bump_cats, all_cats, df)

    Path(OUTPUT_TABLE_CSV).parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(OUTPUT_TABLE_CSV, index=False)
    print(f"  Table saved → {OUTPUT_TABLE_CSV}")

    print("\nSummary (averaged across model-variants):")
    summary = (
        table.groupby('error_category')
             .agg(bump_total=('bump_total', 'first'),
                  avg_llm_detected=('llm_detected', 'mean'),
                  avg_in_common=('in_common', 'mean'),
                  avg_rate=('detection_rate_%', 'mean'))
             .round(1)
             .sort_values('avg_rate', ascending=False)
    )
    print(summary.to_string())

    print("\nStep 4: Printing LaTeX tables...")
    print_latex(table, all_cats)

    print("\nStep 5: Generating plot...")
    plot_coverage(table, all_cats, OUTPUT_PLOT_PNG)
    print("\nDone.")


if __name__ == "__main__":
    main()


Step 1: Loading detected CSV...
  Detected CSV: 137 rows | 32 unique instances | 3 models | 3 variants

Step 2: Loading BUMP CSV (using normalized_error_category)...
  BUMP CSV: 32 instances matched to detected CSV
  Categories present (8):
    NoClassDefFoundError                           (20 instances)
    ClassNotFoundException                         (16 instances)
    ClassCastException                             (7 instances)
    NoSuchMethodError                              (5 instances)
    MojoFailureException                           (3 instances)
    org.json.JSONException                         (1 instances)
    ExceptionInInitializerError                    (1 instances)
    Other                                          (8 instances)

Step 3: Building coverage table...
  Table saved → /Volumes/Rachna-HD/RQResultsForPaper/RQ3/analysis1_error_type_coverage_table.csv

Summary (averaged across model-variants):
                             bump_total  avg_llm_detected  a

In [12]:
"""
add_error_category_to_bump.py
==============================
Adds normalized_error_category column to BUMP CSV.
Reads exception_types, maps each to CATEGORY_MAP, unknown → Other,
Maven exceptions excluded entirely.
Writes to a new file to avoid corrupting the original.
"""

import csv
import pandas as pd
from collections import Counter


# ─── CONFIG ───────────────────────────────────────────────────────────────────
BUMP_CSV   = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes.csv"
OUTPUT_CSV = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes_with_categories.csv"
# ──────────────────────────────────────────────────────────────────────────────

MAVEN_EXCLUDED = {'MojoFailureException', 'MojoExecutionException'}

CATEGORY_MAP: dict[str, str] = {
    'NullPointerException':         'NullPointerException',
    'ClassNotFoundException':       'ClassNotFoundException',
    'NoClassDefFoundError':         'NoClassDefFoundError',
    'NoSuchMethodError':            'NoSuchMethodError',
    'NoSuchMethodException':        'NoSuchMethodException',
    'NoSuchFieldError':             'NoSuchFieldError',
    'AbstractMethodError':          'AbstractMethodError',
    'IncompatibleClassChangeError': 'IncompatibleClassChangeError',
    'ClassCastException':           'ClassCastException',
    'IllegalAccessError':           'IllegalAccessError',
    'IllegalArgumentException':     'IllegalArgumentException',
    'VerifyError':                  'VerifyError',
    'LinkageError':                 'LinkageError',
    'ExceptionInInitializerError':  'ExceptionInInitializerError',
    'AssertionError':               'AssertionError',
    'AssertionFailedError':         'AssertionFailedError',
    'InvocationTargetException':    'InvocationTargetException',
}
OTHER = 'Other'


def categorize_exceptions(raw: str) -> str:
    if not raw or str(raw).strip() in ('', 'nan'):
        return OTHER

    cats_known = []
    has_other  = False

    for e in str(raw).split('|'):
        # Strip package prefix to get short class name
        short = e.strip().split('.')[-1]
        if not short:
            continue
        # Skip Maven exceptions entirely
        if short in MAVEN_EXCLUDED:
            continue
        cat = CATEGORY_MAP.get(short)
        if cat:
            if cat not in cats_known:
                cats_known.append(cat)
        else:
            has_other = True  # unknown → Other

    if not cats_known and not has_other:
        return OTHER

    return '|'.join(cats_known + ([OTHER] if has_other else []))


def detect_sep(path: str) -> str:
    with open(path, encoding='utf-8') as f:
        header = f.readline()
    tabs   = header.count('\t')
    commas = header.count(',')
    sep = '\t' if tabs > commas else ','
    print(f"  Separator: {'TAB' if sep == chr(9) else 'COMMA'} (tabs={tabs}, commas={commas})")
    return sep


def main():
    sep = detect_sep(BUMP_CSV)
    df  = pd.read_csv(BUMP_CSV, sep=sep, dtype=str, keep_default_na=False,
                      quoting=csv.QUOTE_MINIMAL)
    print(f"  Loaded {len(df)} rows, {len(df.columns)} columns")
    print(f"  Columns: {list(df.columns)}")

    # Debug: show raw exception_types for first 5 rows
    print(f"\n  Sample exception_types (first 5 rows):")
    for _, row in df.head(5).iterrows():
        raw = row.get('exception_types', '')
        cat = categorize_exceptions(raw)
        print(f"    {row['custom_id']:8s}  raw: {raw[:80]}")
        print(f"    {'':8s}  cat: {cat}")

    # Add normalized column
    df['normalized_error_category'] = df['exception_types'].apply(categorize_exceptions)

    # Summary
    cat_counts: Counter = Counter()
    for val in df['normalized_error_category']:
        for c in str(val).split('|'):
            cat_counts[c.strip()] += 1
    print(f"\n  Category breakdown ({len(cat_counts)} categories):")
    for cat, cnt in sorted(cat_counts.items(), key=lambda x: -x[1]):
        print(f"    {cat:45s}  {cnt:3d} instances")

    # Write to new file, same separator
    df.to_csv(OUTPUT_CSV, sep=sep, index=False, quoting=csv.QUOTE_MINIMAL)
    print(f"\n  Saved → {OUTPUT_CSV}  ({len(df.columns)} columns, {len(df)} rows)")
    print(f"  Original {BUMP_CSV} is untouched.")


if __name__ == "__main__":
    main()

  Separator: COMMA (tabs=0, commas=30)
  Loaded 89 rows, 31 columns
  Columns: ['custom_id', 'clientGithubURL', 'clientProject', 'clientProjectOrganisation', 'breakingCommit', 'dependencyGroupID', 'dependencyArtifactID', 'previousVersion', 'newVersion', 'failureCategory', 'docker_image_breaking', 'execution_timestamp', 'execution_success', 'return_code', 'execution_time_seconds', 'tests_run', 'test_failures', 'test_errors', 'test_skipped', 'num_exceptions', 'num_failed_tests', 'num_compilation_errors', 'num_error_messages', 'exception_types', 'first_error_message', 'all_maven_errors', 'build_status', 'error_category', 'notes', 'log_file', 'parsed_errors_file']

  Sample exception_types (first 5 rows):
    BBC01     raw: java.lang.NoClassDefFoundError|StopException|java.lang.ClassNotFoundException|Mo
              cat: NoClassDefFoundError|ClassNotFoundException|Other
    BBC02     raw: java.lang.NoClassDefFoundError|java.lang.RuntimeException|java.lang.ClassNotFoun
              cat: N

In [1]:
"""
analysis1_error_type_coverage.py
=================================
Pipeline:
  1. BUMP CSV      → reads normalized_error_category column directly
                     all_cats  = ALL unique categories across the full BUMP CSV
                     bump_total = # instances per category from the FULL BUMP CSV
                                  (NOT restricted to detected CSV — this was the bug)
  2. Detected CSV  → per (model, context_variant, custom_id):
                       llm_detected_errors : what LLM threw
                       matched_errors      : intersection with BUMP errors
  3. For each (model, context_variant, error_category):
       bump_total   = # BUMP instances (full CSV) with this category
       llm_detected = # instances where LLM detected this category
       in_common    = # instances where both BUMP and LLM share this category
       rate         = in_common / bump_total * 100
"""

import pandas as pd
from collections import defaultdict
from pathlib import Path


# ─── CONFIG ───────────────────────────────────────────────────────────────────
BUMP_CSV         = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes_with_categories.csv"
DETECTED_CSV     = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/detected_bc_errortype_coverage.csv"
OUTPUT_TABLE_CSV = "/Volumes/Rachna-HD/RQResultsForPaper/RQ3/analysis1_error_type_coverage_table.csv"

MODEL_ORDER   = ['GPT-4o', 'Qwen-480B', 'GPT-OSS-120b']
VARIANT_ORDER = ['Minimal', 'Method', 'Class']

MAVEN_EXCLUDED = {'MojoFailureException', 'MojoExecutionException'}
OTHER = 'Other'
# ──────────────────────────────────────────────────────────────────────────────


def parse_pipe_raw(val) -> set[str]:
    """Pipe-separated raw error string → set of short class names, Maven excluded."""
    if not val or str(val).strip() in ('', 'nan'):
        return set()
    return {
        e.strip().split('.')[-1]
        for e in str(val).split('|')
        if e.strip() and not any(ex in e for ex in MAVEN_EXCLUDED)
    }


def parse_pipe_cats(val) -> set[str]:
    """Pipe-separated normalized categories → set of category strings."""
    if not val or str(val).strip() in ('', 'nan'):
        return set()
    return {c.strip() for c in str(val).split('|') if c.strip()}


# ─── STEP 1: Load BUMP CSV ────────────────────────────────────────────────────

def load_bump(bump_csv: str, detected_ids: set[str]) -> tuple[dict[str, set[str]], list[str], dict[str, int]]:
    """
    Reads normalized_error_category from the full BUMP CSV.

    bump_total_by_cat : counted from ALL rows in BUMP CSV (the fix)
    bump_cats         : restricted to detected_ids, used for llm_detected / in_common
    all_cats          : ordered by frequency across full BUMP CSV, Other last
    """
    df = pd.read_csv(bump_csv, sep=None, engine='python', dtype=str, keep_default_na=False)
    df['custom_id'] = df['custom_id'].astype(str).str.strip()

    bump_cats: dict[str, set[str]] = {}
    all_counts: dict[str, int] = defaultdict(int)
    detected_counts: dict[str, int] = defaultdict(int)

    for _, row in df.iterrows():
        cid  = row['custom_id']
        cats = parse_pipe_cats(row.get('normalized_error_category', ''))
        # bump_total always from full BUMP CSV
        for c in cats:
            all_counts[c] += 1
        # bump_cats restricted to detected_ids (for llm_detected / in_common)
        if cid in detected_ids:
            bump_cats[cid] = cats
            for c in cats:
                detected_counts[c] += 1

    # Order by frequency descending (full BUMP), Other always last
    all_cats = sorted(
        (c for c in all_counts if c != OTHER), key=lambda c: -all_counts[c]
    ) + ([OTHER] if OTHER in all_counts else [])

    print(f"  BUMP CSV total: {len(df)} instances")
    print(f"  All unique categories ({len(all_cats)}) [full BUMP counts]:")
    for c in all_cats:
        print(f"    {c:45s}  full={all_counts[c]}  detected={detected_counts.get(c,0)}")

    return bump_cats, all_cats, dict(all_counts)


# ─── STEP 2: Load detected CSV ────────────────────────────────────────────────

def load_detected(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=None, engine='python', dtype=str, keep_default_na=False)
    df['custom_id']     = df['custom_id'].astype(str).str.strip()
    df['_llm_cats']     = df['llm_detected_errors'].apply(parse_pipe_raw)
    df['_matched_cats'] = df['matched_errors'].apply(parse_pipe_raw)

    print(f"  Detected CSV: {len(df)} rows | "
          f"{df['custom_id'].nunique()} unique instances | "
          f"{df['model'].nunique()} models | "
          f"{df['context_variant'].nunique()} variants")
    return df


# ─── STEP 3: Build coverage table ────────────────────────────────────────────

def build_table(
    bump_total_by_cat: dict[str, int],
    bump_cats: dict[str, set[str]],
    all_cats: list[str],
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    bump_total   : from full BUMP CSV (bump_total_by_cat)
    llm_detected : # detected-CSV instances where LLM threw this category,
                   counted per custom_id using _llm_cats (raw class names
                   matched against bump_cats category membership)
    in_common    : # instances where both BUMP and LLM agree on this category
                   (i.e. cat in bump_cats[cid] AND cat in _llm_cats[cid])
    """
    records = []
    for model in MODEL_ORDER:
        for variant in VARIANT_ORDER:
            sub = df[(df['model'] == model) & (df['context_variant'] == variant)]

            for cat in all_cats:
                bump_total   = bump_total_by_cat.get(cat, 0)  # from full BUMP CSV

                # llm_detected: within this (model, variant) bucket,
                # simply count rows where LLM threw this category (no BUMP matching)
                llm_detected = int(sub['_llm_cats'].apply(lambda s: cat in s).sum())

                # in_common: within this (model, variant) bucket,
                # count rows where BUMP has this category AND LLM also detected it
                in_common = int(sub.apply(
                    lambda row: cat in bump_cats.get(row['custom_id'], set())
                                and cat in row['_llm_cats'],
                    axis=1
                ).sum())

                rate = round(in_common / bump_total * 100, 1) if bump_total > 0 else 0.0

                records.append({
                    'model':            model,
                    'context_variant':  variant,
                    'error_category':   cat,
                    'bump_total':       bump_total,
                    'llm_detected':     llm_detected,
                    'in_common':        in_common,
                    'detection_rate_%': rate,
                })

    return pd.DataFrame(records)


# ─── STEP 4: LaTeX tables (one per model) ────────────────────────────────────

def print_latex(table: pd.DataFrame, all_cats: list[str]):
    # Only show categories that exist in the full BUMP CSV
    active_cats = [c for c in all_cats if table[table['error_category'] == c]['bump_total'].iloc[0] > 0]

    for model in MODEL_ORDER:
        model_df = table[table['model'] == model]

        print(f"\n% ── Model: {model} ──────────────────────────────────────────────")
        print(r"\begin{table}[htbp]")
        print(r"\centering")
        print(f"\\caption{{Error Type Coverage for \\textbf{{{model}}}}}")
        print(f"\\label{{tab:coverage_{model.replace('-','_').replace(' ','_')}}}")
        print(r"\resizebox{\columnwidth}{!}{%")
        # Columns: category | bump_total | then per variant: LLM, Common, Rate%
        print(r"\begin{tabular}{l r | r r r | r r r | r r r}")
        print(r"\toprule")
        print(r"& & \multicolumn{3}{c|}{\textit{Minimal}} "
              r"& \multicolumn{3}{c|}{\textit{Method}} "
              r"& \multicolumn{3}{c}{\textit{Class}} \\")
        print(r"\cmidrule(lr){3-5}\cmidrule(lr){6-8}\cmidrule(lr){9-11}")
        print(r"\textbf{Error Category} & \textbf{BUMP $n$} "
              r"& LLM & Com & \% "
              r"& LLM & Com & \% "
              r"& LLM & Com & \% \\")
        print(r"\midrule")

        for cat in active_cats:
            bump_n = model_df[model_df['error_category'] == cat]['bump_total'].iloc[0]
            cells = []
            for variant in VARIANT_ORDER:
                r = model_df[
                    (model_df['context_variant'] == variant) &
                    (model_df['error_category'] == cat)
                ]
                llm  = r['llm_detected'].values[0]
                com  = r['in_common'].values[0]
                rate = r['detection_rate_%'].values[0]
                cells.append(f"{llm} & {com} & {rate}")

            print(f"\\texttt{{{cat}}} & {bump_n} & " + " & ".join(cells) + r" \\")

        print(r"\bottomrule")
        print(r"\end{tabular}%")
        print(r"}")
        print(r"\end{table}")


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    print("\nStep 1: Loading detected CSV...")
    df = load_detected(DETECTED_CSV)

    print("\nStep 2: Loading BUMP CSV (bump_total from full CSV, rest filtered to detected)...")
    bump_cats, all_cats, bump_total_by_cat = load_bump(BUMP_CSV, detected_ids=set(df['custom_id'].unique()))

    print("\nStep 3: Building coverage table...")
    table = build_table(bump_total_by_cat, bump_cats, all_cats, df)

    Path(OUTPUT_TABLE_CSV).parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(OUTPUT_TABLE_CSV, index=False)
    print(f"  Table saved → {OUTPUT_TABLE_CSV}")

    print("\nSummary (averaged across model-variants):")
    summary = (
        table[table['bump_total'] > 0]
        .groupby('error_category')
        .agg(
            bump_total=('bump_total', 'first'),
            avg_llm_detected=('llm_detected', 'mean'),
            avg_in_common=('in_common', 'mean'),
            avg_rate=('detection_rate_%', 'mean')
        )
        .round(1)
        .sort_values('avg_rate', ascending=False)
    )
    print(summary.to_string())

    print("\nStep 4: Printing LaTeX tables (one per model)...")
    print_latex(table, all_cats)

    print("\nDone.")


if __name__ == "__main__":
    main()


Step 1: Loading detected CSV...
  Detected CSV: 137 rows | 32 unique instances | 3 models | 3 variants

Step 2: Loading BUMP CSV (bump_total from full CSV, rest filtered to detected)...
  BUMP CSV total: 89 instances
  All unique categories (12) [full BUMP counts]:
    NoClassDefFoundError                           full=35  detected=20
    ClassNotFoundException                         full=30  detected=16
    ClassCastException                             full=10  detected=7
    AssertionError                                 full=10  detected=0
    ExceptionInInitializerError                    full=8  detected=1
    NoSuchMethodError                              full=8  detected=5
    AssertionFailedError                           full=7  detected=0
    AbstractMethodError                            full=4  detected=0
    IllegalArgumentException                       full=3  detected=0
    InvocationTargetException                      full=1  detected=0
    NullPointerException   